# ✈️ TRIP.com Price Crawler — **v1** (freeze / ổn định)

Bản **v1** = snapshot code đang chạy ổn. Không có speed-up. Nếu `v2` lỗi → quay lại notebook này (hoặc `../run_trip.ipynb` ở root).

Chạy lần lượt: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

> ⚠️ Trip chạy **BROWSER-only**. Chậm hơn Agoda nhưng ổn định.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Output** nằm trong `results/trip/`:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_trip.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1g_S06QeEAWnCTHYXGH0Nn4Mcb3FCT_-uIS1jUm4GGkw/edit?gid=607908359#gid=607908359"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
OFFLINE_FILE = "input/trip_hotels.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/2" = chạy phần 1 trong 2 phần (chạy lần lượt 1/2 rồi 2/2)

In [ ]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

_VERSION = 'v1'

def _find_roots(version):
    """Tìm PKG_ROOT (…/vN) + TOOL_ROOT (…/31.crawl-tool) kể cả khi cwd đang ở results/…"""
    cur = os.path.abspath("")
    seen = set()
    for _ in range(10):
        if cur in seen:
            break
        seen.add(cur)
        if os.path.basename(cur) == version and os.path.isdir(os.path.join(cur, "crawler")):
            return cur, os.path.dirname(cur)
        if os.path.isdir(os.path.join(cur, version, "crawler")):
            return os.path.join(cur, version), cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        f"Không tìm thấy {version}/crawler (cwd={os.path.abspath('')!r}). "
        f"Mở notebook từ 31.crawl-tool/ hoặc {version}/, hoặc Restart Kernel rồi chạy lại từ cell ①.")

PKG_ROOT, TOOL_ROOT = _find_roots(_VERSION)
ROOT = TOOL_ROOT  # shared input/; results may be versioned below
os.chdir(TOOL_ROOT)  # ổn định cwd (tránh kẹt ở results/ sau cell crawl trước)

if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

# Prefer this folder's crawler over a previously imported root crawler
if "crawler" in sys.modules:
    del sys.modules["crawler"]
    for k in list(sys.modules):
        if k.startswith("crawler."):
            del sys.modules[k]

import crawler
from crawler.hotels_io import read_hotels
print(f"📦 crawler {_VERSION} @ {PKG_ROOT} | version={getattr(crawler, '__version__', '?')}")
print(f"📂 TOOL_ROOT={TOOL_ROOT} | cwd={os.getcwd()}")

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(TOOL_ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "trip_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/trip_hotels.csv


In [ ]:
# ════════════════ ③ CRAWL ════════════════
_prev_cwd = os.getcwd()
OUTDIR = os.path.join(ROOT, "results", "trip")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)

kwargs = dict(
    site="trip",
    input=INPUT,
    weeks=WEEKS,
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

try:
    await crawler.arun(**kwargs)
finally:
    os.chdir(_prev_cwd)


In [ ]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "trip")
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)
